# 第3课：nn.Module 与神经网络构建

**学习目标：**
- 理解 `nn.Module` 是 PyTorch 神经网络的基础
- 掌握 `nn.Linear` 全连接层
- 用 `nn.Module` 构建多层网络
- 与 NumPy 手写 Layer/Network 类对比

---

在 NumPy 教程中，我们手写了 `Layer` 和 `Network` 类。PyTorch 提供了 `nn.Module` 和 `nn.Linear`，让网络构建变得极其简洁。

## 3.1 nn.Linear：全连接层

`nn.Linear(in_features, out_features)` 等价于我们手写的 `Layer` 类：

$$output = input \cdot W^T + b$$

权重和偏置自动初始化。

In [ ]:
import torch
import torch.nn as nn

# 创建一个全连接层：输入2维，输出3维
linear = nn.Linear(in_features=2, out_features=3)

print("权重形状:", linear.weight.shape)  # (3, 2) — 注意是转置的
print("偏置形状:", linear.bias.shape)    # (3,)

# 前向传播
x = torch.tensor([[1.0, 2.0]])  # (1, 2)
output = linear(x)               # (1, 3)
print("\n输出:", output)

## 3.2 激活函数

PyTorch 提供了所有常用激活函数：

In [ ]:
x = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])

print("ReLU:", torch.relu(x))
print("Sigmoid:", torch.sigmoid(x))
print("Tanh:", torch.tanh(x))

# 也可以用 nn 模块
relu = nn.ReLU()
print("nn.ReLU:", relu(x))

## 3.3 用 nn.Module 构建网络

自定义网络需要继承 `nn.Module` 并实现 `__init__` 和 `forward` 方法。

In [ ]:
class MyNetwork(nn.Module):
    def __init__(self):
        super().__init__()  # 必须调用父类构造函数
        
        # 定义层
        self.layer1 = nn.Linear(2, 16)   # 输入2维 → 16维
        self.layer2 = nn.Linear(16, 16)  # 16维 → 16维
        self.layer3 = nn.Linear(16, 2)   # 16维 → 2维输出
        self.relu = nn.ReLU()
    
    def forward(self, x):
        """前向传播"""
        x = self.relu(self.layer1(x))  # 隐藏层1 + ReLU
        x = self.relu(self.layer2(x))  # 隐藏层2 + ReLU
        x = self.layer3(x)             # 输出层（不加激活）
        return x

# 创建网络
net = MyNetwork()
print(net)  # 打印网络结构

## 3.4 前向传播

In [ ]:
# 3个样本，每个2维
x = torch.tensor([[1.0, 2.0],
                   [3.0, 4.0],
                   [5.0, 6.0]])

# 前向传播
output = net(x)
print("输出形状:", output.shape)  # (3, 2)
print("输出:", output)

## 3.5 查看和管理参数

In [ ]:
# 查看所有参数
total_params = 0
for name, param in net.named_parameters():
    print(f"{name:20s} | shape: {str(param.shape):15s} | requires_grad: {param.requires_grad}")
    total_params += param.numel()

print(f"\n总参数量: {total_params}")

## 3.6 Sequential 快捷写法

对于简单的层堆叠，可以用 `nn.Sequential` 更简洁地定义：

In [ ]:
net_seq = nn.Sequential(
    nn.Linear(2, 16),
    nn.ReLU(),
    nn.Linear(16, 16),
    nn.ReLU(),
    nn.Linear(16, 2)
)

print(net_seq)
print("\n输出:", net_seq(x))

---

## NumPy vs PyTorch 对照

| NumPy 手写 | PyTorch |
| --- | --- |
| `Layer(n_inputs, n_neurons)` | `nn.Linear(n_inputs, n_neurons)` |
| `np.dot(inputs, W) + b` | `layer(inputs)` |
| `np.maximum(0, x)` | `nn.ReLU()(x)` |
| 手写 `Network` 类 | `nn.Module` + `nn.Sequential` |
| 手动 `np.random.randn` 初始化 | 自动初始化（Xavier/Kaiming） |

---

## 小结

- `nn.Linear(in, out)` = 全连接层，自动管理权重和偏置
- 继承 `nn.Module` 实现自定义网络
- `forward()` 定义前向传播逻辑
- `nn.Sequential` 可以快速堆叠简单网络

**下一课**我们将把 Softmax 和分类任务也用 PyTorch 实现。